In [9]:
%load_ext autoreload
%autoreload 2
import os
if not hasattr(__builtins__, '_cwd_set'):
    os.chdir('..')
    __builtins__._cwd_set = True

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# In-memory DCR.js XML round trip

The existing source XML is read once. Every generated XML document then stays in memory as a `str`: no temporary files are created. The complete XML is sent in the JSON `graph_xml` field and returned by the backend after each state transition.

In [10]:
from pathlib import Path
from pprint import pprint

import httpx

from pm4py.objects.dcr.exporter import exporter as dcr_exporter
from pm4py.objects.dcr.importer import importer as dcr_importer
from pm4py.objects.dcr.ocdcr.obj import DcrActivity

In [11]:
def export_xml(graph) -> str:
    # The exporter returns bytes directly; it never writes a file.
    return dcr_exporter.serialize(
        graph, variant=dcr_exporter.DCR_JS_PORTAL
    ).decode('utf-8')


def import_xml(xml: str):
    return dcr_importer.deserialize(
        xml, variant=dcr_importer.DCR_JS_PORTAL
    )


def marking(graph) -> dict:
    return {
        element.ID: {
            'included': element.included,
            'pending': element.pending,
            'executed': element.executed is not None,
            'data': element.data,
        }
        for element in sorted(graph.elements, key=lambda item: item.ID)
        if isinstance(element, DcrActivity)
    }


def marking_difference(before: dict, after: dict) -> list[dict]:
    changes = []
    for activity_id in sorted(before.keys() | after.keys()):
        old, new = before.get(activity_id), after.get(activity_id)
        if old != new:
            changes.append({'activity': activity_id, 'before': old, 'after': new})
    return changes

## 1. Local XML → graph → XML → graph

An empty difference proves that importing and exporting preserved the marking.

In [12]:
MODEL_PATH = Path('data/models/Social Service Law 86 Data EN.xml')
source_xml = MODEL_PATH.read_text(encoding='utf-8')
source_graph = import_xml(source_xml)
source_marking = marking(source_graph)

exported_xml = export_xml(source_graph)
local_roundtrip_graph = import_xml(exported_xml)
local_roundtrip_marking = marking(local_roundtrip_graph)

print('Generated XML characters:', len(exported_xml))
print('Marking difference after local round trip:')
pprint(marking_difference(source_marking, local_roundtrip_marking))

Generated XML characters: 13634
Marking difference after local round trip:
[]


## 2. Send the XML inside the JSON body

Start the FastAPI backend on `127.0.0.1:8000` before running the next cells. The initial response returns the backend's in-memory graph as XML.

In [13]:
API_URL = 'http://127.0.0.1:8000/api/chat/response'
DCR_CHAT = 1
request_body = {
    'text': 'Start the DCR session.',
    'chat_type': DCR_CHAT,
    'graph_xml': exported_xml,  # Complete XML string in the JSON body.
}
print({**request_body, 'graph_xml': f'<{len(exported_xml)} XML characters>'})

first_response = httpx.post(API_URL, json=request_body, timeout=30)
first_response.raise_for_status()
first_body = first_response.json()
session_id = first_body['session_id']
backend_graph = import_xml(first_body['graph_xml'])
backend_marking = marking(backend_graph)

print('Marking difference after client → backend → client round trip:')
pprint(marking_difference(local_roundtrip_marking, backend_marking))
print('Next enabled activity:', first_body['act_id'], first_body['text'])

{'text': 'Start the DCR session.', 'chat_type': 1, 'graph_xml': '<13634 XML characters>'}
Marking difference after client → backend → client round trip:
[]
Next enabled activity: Event_0muqobt Not covered by other laws


## 3. Execute one activity and round-trip the updated marking

The session remains in backend memory. Its response contains fresh XML, which is imported directly from the response body and compared with the previous marking.

In [14]:
execution_body = {
    'text': 'No',
    'session_id': session_id,
    'act_id': first_body['act_id'],
}
execution_response = httpx.post(API_URL, json=execution_body, timeout=30)
execution_response.raise_for_status()
execution_result = execution_response.json()

In [16]:
HISTORY_URL = 'http://127.0.0.1:8000/api/chat/history'
hist_response = httpx.post(HISTORY_URL, json={"session_id":session_id}, timeout=30)
hist_response.json()

[{'item': 'Start the DCR session.', 'chat_role': 'user', 'dcr_role': None},
 {'item': 'Not covered by other laws',
  'chat_role': 'assistant',
  'dcr_role': 'Citizen Data'},
 {'item': 'No', 'chat_role': 'user', 'dcr_role': None},
 {'item': 'Consequence of disability',
  'chat_role': 'assistant',
  'dcr_role': 'Citizen Data'}]

In [ ]:
execution_result

In [ ]:

updated_graph = import_xml(execution_result['graph_xml'])
updated_marking = marking(updated_graph)

print('Marking difference after backend execution and XML round trip:')
pprint(marking_difference(backend_marking, updated_marking))
print('Next enabled activity:', execution_result['act_id'], execution_result['text'])

In [ ]:
delete_response = httpx.request(
    'DELETE',
    'http://127.0.0.1:8000/api/chat/session',
    json={'session_id': first_body['session_id']},
)
delete_response.raise_for_status()
print('Session removed from backend memory.')